**1. Import Libraries**

In [ ]:
!pip install rouge-score bert-score --quiet

# libraries
import os
import json
import torch
import zipfile
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from rouge_score import rouge_scorer
from bert_score import score as bert_score_compute

# mount google drive
from google.colab import drive
drive.mount('/content/drive')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.8 MB/s eta 0:00:00
Mounted at /content/drive


**2. Load MedInstruct Dataset**

In [ ]:
# folder path to the medical dataset files
folder_path = "/content/drive/MyDrive/Fine-Tuning-Datasets/MedInstruct"

# load the test set
test_df = pd.read_json(f"{folder_path}/medical_test.jsonl", lines=True)

# print dataset info
print(f"Test Dataset Size: {len(test_df)}")
print(f"\nColumn names: {test_df.columns.tolist()}")
print(f"\nFirst example:\n{test_df.iloc[0]}")

Test Dataset Size: 2965

Column names: ['instruction', 'input', 'output']

First example:
instruction    Given a medical scenario and a question about ...
input          What is the diagnosis for a patient with high ...
output         The diagnosis for a patient with high levels o...
Name: 0, dtype: object


**3. Random Sample 500 Rows for Testing**

In [ ]:
# randomly sample 500 examples
test_subset = test_df.sample(n=500, random_state=42).reset_index(drop=True)

print(f"Subset size: {len(test_subset)}")
print(f"\nFirst example:")
print(f"Instruction: {test_subset.iloc[0]['instruction']}")
print(f"Input:       {test_subset.iloc[0]['input']}")
print(f"Output:      {test_subset.iloc[0]['output']}")

Subset size: 500

First example:
Instruction: Recommend appropriate physical therapy exercises for a patient recovering from a stroke with limited mobility in their left arm.
Input:       Patient X suffered a stroke three months ago, resulting in limited mobility and muscle weakness in the left arm. The patient can slightly bend the elbow and wrist but experiences difficulty in lifting objects or fully extending the arm.
Output:      Passive range of motion exercises, isometric exercises, biceps curls with light resistance, assisted arm stretches, wall push-ups, seated table slides, and shoulder abduction with a resistance band.


**4. Load Model Adapters From Google Drive**

In [ ]:
# path to the model zip file in google drive
zip_path = "/content/drive/MyDrive/Fine-Tuning-Notebooks/Low-Rank-Adaptation-(LoRA)/Llama-3.2-3B-Instruct/Medical/medical_llama3_lora_adapter.zip"

# where to unzip the model in colab
output_path = "/content/model"

# unzips the model into the colab workspace
print("Unzipping model...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(output_path)

# confirms the model has been unzipped and shows the folder contents
print(f"Model unzipped to {output_path}")
print(os.listdir(output_path))

Unzipping model...
Model unzipped to /content/model
['medical_llama3_lora_adapter']


**5. Load Model Llama-3.2-3B-Instruct and apply adapters (Medical)**

In [ ]:
# path to the unzipped lora adapter
adapter_path = "/content/model/medical_llama3_lora_adapter"

# base model name from hugging face
base_model_name = "meta-llama/Llama-3.2-3B-Instruct"

# loads the tokenizer from the base model
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# add padding token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# loads the base model
print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)

# loads the lora adapter on top of the base model
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

# check if the lora adapter loaded correctly
if isinstance(model, PeftModel):
    print("LoRA adapter is active!")
else:
    print("Warning: adapter did not load, running base model only.")

print("Model loaded and ready!")

Loading tokenizer...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Loading LoRA adapter...
LoRA adapter is active!
Model loaded and ready!


**6. Perform Evaluation (MedInstruct)**

In [ ]:
# set seed for reproducibility
torch.manual_seed(42)

# get device from model
device = next(model.parameters()).device

# join instruction and input
def format_prompt(row):
    if pd.notna(row['input']) and row['input'].strip():
        return f"{row['instruction']}\n\n{row['input']}"
    return row['instruction']

# build chat template
def build_chat_prompt(tokenizer, prompt):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return formatted

# turn prompt to tokens
def tokenize_prompt(tokenizer, formatted_prompt, device):
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
    return inputs

# generate tokens
def generate_tokens(model, tokenizer, inputs, max_new_tokens=2048):
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return outputs

# get response text
def decode_response(tokenizer, outputs, input_length):
    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )
    return response

# run full generation
def generate_response(model, tokenizer, prompt, device, max_new_tokens=2048):
    formatted_prompt = build_chat_prompt(tokenizer, prompt)
    inputs = tokenize_prompt(tokenizer, formatted_prompt, device)
    input_length = inputs["input_ids"].shape[1]
    outputs = generate_tokens(model, tokenizer, inputs, max_new_tokens=max_new_tokens)
    response = decode_response(tokenizer, outputs, input_length)
    return response

# get predictions and references
def collect_predictions(model, tokenizer, test_subset, device):
    predictions = []
    references = []

    for idx, (_, row) in enumerate(tqdm(test_subset.iterrows(), total=len(test_subset), desc="Evaluating"), 1):
        prompt = format_prompt(row)
        response = generate_response(model, tokenizer, prompt, device)

        predictions.append(response.strip())
        references.append(row['output'].strip())

        if idx % 50 == 0:
            print(f"Completed {idx}/{len(test_subset)} examples")

    return predictions, references

# get rouge-l score
def compute_rouge_l(predictions, references):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = [
        scorer.score(ref, pred)['rougeL'].fmeasure
        for pred, ref in zip(predictions, references)
    ]
    avg_rouge = np.mean(rouge_scores)
    return avg_rouge

# get bertscore f1
def compute_bertscore_f1(predictions, references, device):
    P, R, F1 = bert_score_compute(
        predictions, references,
        lang="en",
        model_type="roberta-large",
        device=device
    )
    avg_bertscore = F1.mean().item()
    return avg_bertscore

# save predictions to file
def save_predictions(model_name, predictions, references):
    output_dir = "/content/eval_results"
    os.makedirs(output_dir, exist_ok=True)

    safe_name = model_name.replace(" ", "_").lower()
    output_path = f"{output_dir}/{safe_name}_predictions.jsonl"

    with open(output_path, "w") as f:
        for pred, ref in zip(predictions, references):
            json.dump({"prediction": pred, "reference": ref}, f)
            f.write("\n")

    print(f"Predictions saved to {output_path}")

# show final results
def print_results(model_name, predictions, avg_rouge, avg_bertscore):
    print("\n" + "=" * 50)
    print(f"EVALUATION COMPLETE — {model_name}")
    print(f"Total examples:  {len(predictions)}")
    print(f"ROUGE-L:         {avg_rouge:.4f}")
    print(f"BERTScore F1:    {avg_bertscore:.4f}")
    print("=" * 50)

# run evaluation
def evaluate_model(model, tokenizer, test_subset, device, model_name):
    predictions, references = collect_predictions(model, tokenizer, test_subset, device)

    print("Computing ROUGE-L...")
    avg_rouge = compute_rouge_l(predictions, references)

    print("Computing BERTScore...")
    avg_bertscore = compute_bertscore_f1(predictions, references, device)

    print_results(model_name, predictions, avg_rouge, avg_bertscore)
    save_predictions(model_name, predictions, references)

    return {
        "model": model_name,
        "rouge_l": avg_rouge,
        "bertscore_f1": avg_bertscore
    }

# run evaluation
results = evaluate_model(model, tokenizer, test_subset, device, "LLaMA Medical LoRA")

Evaluating:   0%|          | 0/500 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Completed 50/500 examples
Completed 100/500 examples
Completed 150/500 examples
Completed 200/500 examples
Completed 250/500 examples
Completed 300/500 examples
Completed 350/500 examples
Completed 400/500 examples
Completed 450/500 examples
Completed 500/500 examples
Computing ROUGE-L...
Computing BERTScore...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



EVALUATION COMPLETE — LLaMA Medical LoRA
Total examples:  500
ROUGE-L:         0.4269
BERTScore F1:    0.9169
Predictions saved to /content/eval_results/llama_medical_lora_predictions.jsonl
